# LOCAL BOUNDED VALIDATION COPY

Drive and install cells are replaced. Evaluation uses deliberately near-cap fixtures, not complete-game strength tests. Notebook 02 skips training. Temporary paths in outputs are not the production Colab paths.

# No RL 02 - Optional Supervised Learning

Optional: train the existing policy/value architecture against labelled positions, without self-play. This is supervised deep learning, not a requirement for the classical baseline. Set variant to `supervised` before running. An A100/CUDA runtime accelerates this optional notebook only; inference remains CPU.

## Mount Google Drive

In [1]:
print("Local validation only: no Drive mount.")

Local validation only: no Drive mount.


## Project Setup

Uses `configs/no_rl.yaml`. No league, self-play, PPO, or DQN is run. Colab's installed PyTorch is preserved.

In [2]:
from pathlib import Path
import sys, subprocess
from IPython.display import display
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
def show_figure(fig):
    display(fig)
    plt.close(fig)
PROJECT_ROOT = Path('/mnt/d/Benjamin Data/Introduction to AI/Chess Agent Competition/colab_chess_research/results/no_rl_notebook_validation/fixture-pp15y2lv')
sys.path.insert(0, str(PROJECT_ROOT))
from chess_rl.non_rl import load_no_rl_config
from chess_rl.reproducibility import read_json, sha256
cfg = load_no_rl_config(PROJECT_ROOT)
cfg["non_rl"]["variant"] = "classical"
cfg["run_id"] = "no_rl_local_fixture"
cfg["evaluation"].update(development_games=2, heldout_games=2, bootstrap_samples=100)
cfg["non_rl"]["tuning"].update(trials=1, games_per_trial=2, confirmation_games=2)
print("LOCAL FIXTURE: CPU, one ply before game cap, two games per opponent.")
print("No Drive mount, no training, no export/compliance execution.")
from chess_rl import plots
original_plot_matches = plots.plot_matches
def fixture_plot(root, run_id, summaries, name):
    fig = original_plot_matches(root, run_id, summaries, name)
    fig.axes[0].set_title('LOCAL FIXTURE ONLY: one ply before the game cap')
    fig.axes[0].set_xlabel('Score (%); intervals unavailable with one opening pair')
    plots.save(fig, root, run_id, name)
    return fig
plots.plot_matches = fixture_plot


LOCAL FIXTURE: CPU, one ply before game cap, two games per opponent.
No Drive mount, no training, no export/compliance execution.


## Training Choice

Every expensive cell is disabled for the default classical variant. The default supervised model has six 128-channel residual blocks, a 4,672-action policy head, and one value output. Defaults: up to 20 epochs, AdamW, learning rate 0.0003, cosine scheduling, and validation early stopping.

In [3]:
TRAIN_SUPERVISED = cfg["non_rl"]["variant"] == "supervised"
print("Supervised training enabled:", TRAIN_SUPERVISED)
if TRAIN_SUPERVISED:
    from chess_rl.reproducibility import seed_all, resolve_device
    seed_all(cfg["seed"], cfg["deterministic"])
    print("Training device:", resolve_device(cfg["device"]))
    print("Architecture:", cfg["model"])
else:
    print("Skip this notebook. Open no-RL notebook 03.")

Supervised training enabled: False
Skip this notebook. Open no-RL notebook 03.


## Offline Teacher

Stockfish is used only to label offline data. It is not shipped in the candidate. The configured classical teacher is also supported. This setup runs only for the supervised variant.

In [4]:
if TRAIN_SUPERVISED:
    teacher_path = Path(cfg["dataset"]["engine_path"])
    if cfg["dataset"]["teacher"] == "stockfish" and not teacher_path.is_file():
        subprocess.check_call(["apt-get", "update", "-qq"])
        subprocess.check_call(["apt-get", "install", "-y", "-qq", "stockfish"])
    if cfg["dataset"]["teacher"] == "stockfish":
        if not teacher_path.is_file():
            raise FileNotFoundError(f"Configure dataset.engine_path: {teacher_path}")
        print("Teacher hash:", sha256(teacher_path))

## Build or Resume the Dataset

Provide PGNs or JSONL through a `dataset` override in `configs/no_rl.yaml`. Otherwise the workflow generates and labels legal-playout positions, requesting 100,000 unique examples. This is not a curated strategy curriculum. Source-game splits and held-out exclusion use the shared dataset implementation.

In [5]:
if TRAIN_SUPERVISED:
    from chess_rl.dataset import prepare_dataset, prepare_openings
    prepare_openings(PROJECT_ROOT, seed=cfg["seed"])
    dataset_manifest = prepare_dataset(PROJECT_ROOT, cfg)
    print("Actual split counts:", dataset_manifest["counts"])
    print("Target reached:", dataset_manifest["target_reached"])

## Train and Save

Policy targets are legal teacher moves; value targets are teacher evaluations or completed-game results, with their sources distinguished. Training uses legal-only cross-entropy plus value Huber loss. Latest/best checkpoints, optimizer state, RNG, config, and CSV logs are saved. Interrupted epochs resume from the last completed epoch.

In [6]:
if TRAIN_SUPERVISED:
    from chess_rl.training import fit_supervised
    checkpoint = fit_supervised(PROJECT_ROOT, cfg, dataset_manifest)
    print("Best supervised checkpoint:", checkpoint)
    print("SHA256:", sha256(checkpoint))

## Inspect Learning Curves

These plots use actual completed-epoch logs only. Lower label loss does not establish stronger play. Move on to no-RL notebook 03 to measure games; no self-play stage follows this notebook.

In [7]:
if TRAIN_SUPERVISED:
    from chess_rl.plots import plot_supervised
    show_figure(plot_supervised(PROJECT_ROOT, cfg["run_id"]))